# 🚀 ANN 2026 — Multimodal Crowdfunding Outcome Prediction

**Competition:** ANN 2026 Multimodal Challenge — Crowdfunding Outcome Prediction (Kaggle)

**Goal:** Predict the `target_band` (5-class) for crowdfunding projects using a **multimodal** approach combining:
- 📋 **Tabular features** (category, goal, duration, launch date, creator history, etc.)
- 💬 **Text features** (project title + blurb) via TF-IDF or DeBERTa
- 🖼️ **Image features** (project images) via EfficientNet-B0

**Metric:** Macro F1 Score

---

## 🗺️ Pipeline Overview

| Step | Approach | Notes |
|------|----------|-------|
| 1 | CatBoost (tabular only) | Baseline |
| 2 | DeBERTa-v3-base (text only) | HuggingFace Trainer |
| 3 | DeBERTa + CatBoost blend | 70% text + 30% tabular |
| 4 | TF-IDF + Logistic Regression | Fast text baseline |
| 5 | TF-IDF LR + CatBoost blend | 65% text + 35% tabular |
| 6 | EfficientNet-B0 image features | Pretrained, no fine-tuning |
| 7 | Full Multimodal CatBoost | Tabular + Image + TF-IDF |
| 8 | 5-Fold CV (TF-IDF LR) | OOF validation |


## 📦 Setup & Imports

Install required libraries and import all necessary packages for the pipeline.


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

#import numpy as np # linear algebra
#import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

#import os
#for dirname, _, filenames in os.walk('/kaggle/input'):
 #   for filename in filenames:
  #      print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
!pip install -q transformers timm catboost albumentations

## ⚙️ Configuration

Define all hyperparameters and paths in a central `CFG` class.


In [27]:
import os
import gc
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel

In [28]:
class CFG:

    seed = 42

    train_csv = "/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/train.csv"

    test_csv = "/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/test.csv"

    image_dir = "/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/images_224"

    text_model = "microsoft/deberta-v3-base"

    batch_size = 8

    epochs = 2

    lr = 2e-5

    max_len = 128

    n_splits = 5

    device = torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )

print(CFG.device)

cuda


## 📂 Data Loading

Load training and test CSVs, and verify that all paths exist.


In [29]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/competitions
/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction
/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition
/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed
/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/images_224
/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/images
/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/images/images_224
/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/raw
/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/raw/im

In [31]:
import os

print(os.path.exists(CFG.train_csv))
print(os.path.exists(CFG.test_csv))
print(os.path.exists(CFG.image_dir))

True
True
True


## 🛠️ Feature Engineering

Create text features by combining title + blurb, and add numeric features like log-goal and text lengths.


In [32]:
train_df = pd.read_csv(CFG.train_csv)
test_df = pd.read_csv(CFG.test_csv)

print(train_df.shape)
print(test_df.shape)

train_df.head()

(8870, 19)
(3130, 18)


,project_id,title,blurb,category_main,category_sub,country,currency,goal_usd,duration_days,launch_month,launch_weekday,launch_hour_local,creator_num_prior_projects,creator_prior_success_rate,num_reward_tiers,has_video,description_length,image_path,target_band
0,P004133,Affordable Puzzle,A clean card games project that elevates artis...,Games,Card Games,CA,CAD,33145.46,28,1,6,0,0,0.0,8,1,134,images_224/P004133.jpg,4
1,P003029,Compact Platform,A technical edtech project that helps teachers...,Education,EdTech,UK,GBP,10958.88,45,1,6,1,0,0.0,3,0,105,images_224/P003029.jpg,3
2,P003262,Portable Monitor,A minimal mental wellness project that helps d...,Health,Mental Wellness,US,USD,16452.99,35,1,6,4,0,0.0,5,1,134,images_224/P003262.jpg,4
3,P008745,Next-Gen Companion,A technical nutrition project that reinvents g...,Health,Nutrition,US,USD,150000.00,20,1,6,5,1,1.0,7,0,123,images_224/P008745.jpg,2
4,P000423,Smart Workspace,A retro home project that streamlines artists....,Design,Home,DE,EUR,24911.26,45,1,6,7,0,0.0,4,0,111,images_224/P000423.jpg,4


## 🔢 Label Encoding

Encode categorical columns using `LabelEncoder` fitted on the full (train + test) data.


In [ ]:
def create_features(df):

    df["text"] = (
        df["title"].fillna("") +
        " [SEP] " +
        df["blurb"].fillna("")
    )

    df["log_goal"] = np.log1p(df["goal_usd"])

    df["title_len"] = df["title"].fillna("").apply(len)

    df["blurb_len"] = df["blurb"].fillna("").apply(len)

    return df

train_df = create_features(train_df)
test_df = create_features(test_df)

TARGET = "target_band"

## 🌲 Baseline: CatBoost on Tabular Features

Train a CatBoost multiclass classifier on tabular features only, then generate a baseline submission.


In [ ]:
cat_cols = [
    "category_main",
    "category_sub",
    "country",
    "currency",
    "launch_month",
    "launch_weekday"
]

for col in cat_cols:

    le = LabelEncoder()

    full = pd.concat([
        train_df[col].astype(str),
        test_df[col].astype(str)
    ])

    le.fit(full)

    train_df[col] = le.transform(
        train_df[col].astype(str)
    )

    test_df[col] = le.transform(
        test_df[col].astype(str)
    )

print("Encoding done")

In [ ]:
tabular_features = [
    "category_main",
    "category_sub",
    "country",
    "currency",
    "log_goal",
    "duration_days",
    "launch_month",
    "launch_weekday",
    "launch_hour_local",
    "creator_num_prior_projects",
    "creator_prior_success_rate",
    "num_reward_tiers",
    "has_video",
    "description_length",
    "title_len",
    "blurb_len"
]

X_train = train_df[tabular_features]
y_train = train_df[TARGET]

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    loss_function="MultiClass",
    verbose=100
)

model.fit(X_train, y_train)

preds = model.predict(
    test_df[tabular_features]
)

preds = preds.astype(int).flatten()

print(preds[:10])

## 💬 Text Model: DeBERTa Fine-tuning (HuggingFace Trainer)

Fine-tune `microsoft/deberta-v3-base` on the combined title+blurb text for sequence classification.


In [ ]:
submission = pd.DataFrame({
    "project_id": test_df["project_id"],
    "target_band": preds
})

submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False
)

print(submission.head())
print("submission created!")

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from datasets import Dataset

In [ ]:
train_df["text"] = (
    train_df["title"].fillna("") +
    " " +
    train_df["blurb"].fillna("")
)

test_df["text"] = (
    test_df["title"].fillna("") +
    " " +
    test_df["blurb"].fillna("")
)

In [39]:
model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [40]:
train_dataset = Dataset.from_pandas(
    train_df[["text", "target_band"]]
)

test_dataset = Dataset.from_pandas(
    test_df[["text"]]
)

In [41]:
def tokenize(batch):

    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/8870 [00:00<?, ? examples/s]

Map:   0%|          | 0/3130 [00:00<?, ? examples/s]

In [42]:
train_dataset = train_dataset.rename_column(
    "target_band",
    "labels"
)

In [44]:

train_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_dataset.set_format(
    "torch",
    columns=["input_ids", "attention_mask"]
)

In [46]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=5
)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.bias          

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",

    learning_rate=2e-5,

    per_device_train_batch_size=8,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=100,

    save_strategy="no"
)

In [47]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset
)

In [48]:
trainer.train()

Step,Training Loss
100,1212.788906
200,0.000000
300,0.000000
400,0.000000
500,0.000000
600,0.000000
700,0.000000
800,0.000000
900,0.000000
1000,0.000000


TrainOutput(global_step=1110, training_loss=109.26026182432433, metrics={'train_runtime': 261.4013, 'train_samples_per_second': 67.865, 'train_steps_per_second': 4.246, 'total_flos': 1166949888783360.0, 'train_loss': 109.26026182432433, 'epoch': 2.0})

In [ ]:
predictions = trainer.predict(test_dataset)

## 🔀 Ensemble: DeBERTa + CatBoost (Weighted Blend)

Blend DeBERTa text probabilities (70%) with CatBoost tabular probabilities (30%) for a stronger submission.


In [ ]:
text_probs = torch.softmax(
    torch.tensor(predictions.predictions),
    dim=1
).numpy()

In [ ]:
preds = np.argmax(
    predictions.predictions,
    axis=1
)

In [ ]:
submission = pd.DataFrame({
    "project_id": test_df["project_id"],
    "target_band": preds
})

submission.to_csv(
    "/kaggle/working/submission_text.csv",
    index=False
)

print(submission.head())

In [ ]:
tab_preds = model.predict_proba(
    test_df[tabular_features]
)

In [ ]:
final_probs = (
    0.70 * text_probs +
    0.30 * tab_preds
)

final_preds = np.argmax(
    final_probs,
    axis=1
)

## 📊 Text Baseline: TF-IDF + Logistic Regression

Use TF-IDF bigrams with Logistic Regression as a fast, lightweight text baseline.


In [ ]:
submission = pd.DataFrame({
    "project_id": test_df["project_id"],
    "target_band": final_preds
})

submission.to_csv(
    "/kaggle/working/submission_blend.csv",
    index=False
)

print(submission.head())

In [49]:
predictions = trainer.predict(test_dataset)

In [50]:
import numpy as np

text_probs = torch.softmax(
    torch.tensor(predictions.predictions),
    dim=1
).numpy()

preds = np.argmax(
    text_probs,
    axis=1
)

print(preds[:20])

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
submission = pd.DataFrame({
    "project_id": test_df["project_id"],
    "target_band": preds
})

submission.to_csv(
    "/kaggle/working/submission_deberta.csv",
    index=False
)

print("DONE!")

In [51]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

## 🔀 Ensemble: TF-IDF LR + CatBoost (Blend)

Blend TF-IDF LR probabilities (65%) with CatBoost tabular probabilities (35%).


In [52]:
train_texts = (
    train_df["title"].fillna("") +
    " " +
    train_df["blurb"].fillna("")
)

test_texts = (
    test_df["title"].fillna("") +
    " " +
    test_df["blurb"].fillna("")
)

In [53]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words="english"
)

X_train_text = tfidf.fit_transform(train_texts)
X_test_text = tfidf.transform(test_texts)

In [54]:
text_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

text_model.fit(
    X_train_text,
    train_df["target_band"]
)

LogisticRegression(class_weight='balanced', max_iter=2000)

In [55]:
text_probs = text_model.predict_proba(
    X_test_text
)

text_preds = np.argmax(
    text_probs,
    axis=1
)

print(text_preds[:20])

[4 3 4 2 1 4 1 2 4 3 0 2 1 1 0 3 4 3 2 2]


In [56]:
tab_probs = model.predict_proba(
    test_df[tabular_features]
)

AttributeError: 'DebertaV2ForSequenceClassification' object has no attribute 'predict_proba'

## 🖼️ Image Feature Extraction (EfficientNet-B0)

Extract 1280-dim image embeddings using a pretrained EfficientNet-B0 from `timm`.


In [57]:
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.03,
    depth=6,
    loss_function="MultiClass",
    verbose=False
)

cat_model.fit(
    train_df[tabular_features],
    train_df["target_band"]
)

print("CatBoost trained!")

CatBoost trained!


In [58]:
tab_probs = cat_model.predict_proba(
    test_df[tabular_features]
)

print(tab_probs.shape)

(3130, 5)


In [60]:
final_probs = (
    0.65 * text_probs +
    0.35 * tab_probs
)

final_preds = np.argmax(
    final_probs,
    axis=1
)

In [61]:
submission = pd.DataFrame({
    "project_id": test_df["project_id"],
    "target_band": final_preds
})

submission.to_csv(
    "/kaggle/working/submission_final.csv",
    index=False
)

print(submission.head())
print("DONE!")

  project_id  target_band
0    P000955            4
1    P010897            3
2    P003659            4
3    P003271            3
4    P001407            1
DONE!


In [62]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

In [49]:
train_df["full_text"] = (
    train_df["title"].fillna("") +
    " " +
    train_df["blurb"].fillna("")
)

test_df["full_text"] = (
    test_df["title"].fillna("") +
    " " +
    test_df["blurb"].fillna("")
)

In [66]:
tfidf = TfidfVectorizer(
    max_features=100000,
    ngram_range=(1,2),
    stop_words="english"
)

X = tfidf.fit_transform(train_texts)
X_test = tfidf.transform(test_texts)

y = train_df["target_band"]

In [73]:
text_features = ["full_text"]

In [74]:
from catboost import Pool

train_pool = Pool(
    train_df[features],
    label=train_df["target_band"],
    text_features=text_features
)

test_pool = Pool(
    test_df[features],
    text_features=text_features
)

cat_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    verbose=100
)

cat_model.fit(train_pool)

0:	learn: 0.3563312	total: 982ms	remaining: 16m 20s
100:	learn: 0.4354328	total: 1m 32s	remaining: 13m 44s
200:	learn: 0.4503662	total: 3m 4s	remaining: 12m 11s
300:	learn: 0.4643893	total: 4m 35s	remaining: 10m 39s
400:	learn: 0.4825482	total: 6m 7s	remaining: 9m 8s
500:	learn: 0.5026523	total: 7m 38s	remaining: 7m 36s
600:	learn: 0.5219789	total: 9m 10s	remaining: 6m 5s
700:	learn: 0.5409762	total: 10m 42s	remaining: 4m 33s
800:	learn: 0.5623000	total: 12m 13s	remaining: 3m 2s
900:	learn: 0.5828661	total: 13m 45s	remaining: 1m 30s
999:	learn: 0.6034115	total: 15m 16s	remaining: 0us


CatBoostClassifier(depth=8, eval_metric='TotalF1', iterations=1000, learning_rate=0.03, loss_function='MultiClass', verbose=100)

In [75]:
preds = cat_model.predict(test_pool)

preds = preds.astype(int).flatten()

print(preds[:20])

[4 4 4 3 4 4 1 4 4 1 1 3 1 1 1 4 4 1 1 1]


In [76]:
submission = pd.DataFrame({
    "project_id": test_df["project_id"],
    "target_band": preds
})

submission.to_csv(
    "/kaggle/working/submission_catboost_text.csv",
    index=False
)

print("DONE!")

DONE!


## 🔀 Full Multimodal Fusion: Tabular + Image + TF-IDF

Combine all modalities (tabular, image embeddings, TF-IDF text) into a single CatBoost model.


In [67]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof = np.zeros((len(train_df), 5))
test_preds = np.zeros((len(test_df), 5))

for fold, (train_idx, valid_idx) in enumerate(
    skf.split(X, y)
):

    print(f"FOLD {fold}")

    X_train = X[train_idx]
    y_train = y.iloc[train_idx]

    X_valid = X[valid_idx]
    y_valid = y.iloc[valid_idx]

    clf = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=2.0
    )

    clf.fit(X_train, y_train)

    valid_probs = clf.predict_proba(X_valid)

    oof[valid_idx] = valid_probs

    score = f1_score(
        y_valid,
        np.argmax(valid_probs, axis=1),
        average="macro"
    )

    print("Fold F1:", score)

    test_preds += clf.predict_proba(X_test) / 5

FOLD 0
Fold F1: 0.24771055541674275
FOLD 1
Fold F1: 0.2513958264643853
FOLD 2
Fold F1: 0.2325878503467044
FOLD 3
Fold F1: 0.2421538265833572
FOLD 4
Fold F1: 0.24321959310614938


In [68]:
cv_score = f1_score(
    y,
    np.argmax(oof, axis=1),
    average="macro"
)

print("CV SCORE:", cv_score)

CV SCORE: 0.24342577320147457


In [1]:
import timm
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

In [2]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

image_model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=0
)

image_model = image_model.to(device)
image_model.eval()

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2

In [3]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
])

## ✅ Cross-Validation with TF-IDF + Logistic Regression

Run 5-fold stratified cross-validation to get a reliable OOF F1 score estimate.


In [19]:
import os

In [20]:
def extract_features(df):

    features = []

    for path in tqdm(df["image_path"]):

        img_path = os.path.join(
            CFG.image_dir,
            path
        )

        try:

            image = Image.open(img_path).convert("RGB")

            image = transform(image).unsqueeze(0).to(device)

            with torch.no_grad():

                feat = image_model(image)

            feat = feat.cpu().numpy().flatten()

        except:

            feat = np.zeros(1280)

        features.append(feat)

    return np.array(features)

In [21]:
import pandas as pd

train_df = pd.read_csv(
    "/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/train.csv"
)

test_df = pd.read_csv(
    "/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/test.csv"
)

print(train_df.shape)
print(test_df.shape)

(8870, 19)
(3130, 18)


In [36]:
train_img_features = extract_features(train_df)

100%|██████████| 8870/8870 [02:11<00:00, 67.53it/s]


In [38]:
test_img_features = extract_features(test_df)

100%|██████████| 3130/3130 [00:45<00:00, 68.54it/s]


## 🐛 Debugging: Image Path Verification

Verify that image paths exist correctly before running the full image extraction pipeline.


In [41]:
for i in range(train_img_features.shape[1]):

    train_df[f"img_{i}"] = train_img_features[:, i]

    test_df[f"img_{i}"] = test_img_features[:, i]

In [55]:
features = [
    "category_main",
    "category_sub",
    "country",
    "currency",
    "goal_usd",
    "duration_days",
    "launch_month",
    "launch_weekday",
    "launch_hour_local",
    "creator_num_prior_projects",
    "creator_prior_success_rate",
    "num_reward_tiers",
    "has_video",
    "description_length"
]

In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=3000,
    stop_words="english"
)

train_text = tfidf.fit_transform(
    train_df["full_text"].fillna("")
).toarray()

test_text = tfidf.transform(
    test_df["full_text"].fillna("")
).toarray()

In [58]:
for i in range(train_text.shape[1]):
    train_df[f"tfidf_{i}"] = train_text[:, i]
    test_df[f"tfidf_{i}"] = test_text[:, i]

text_cols = [
    f"tfidf_{i}"
    for i in range(train_text.shape[1])
]

In [61]:


final_features = features + image_cols + text_cols

print(len(final_features))

1492


In [51]:
train_df[final_features].dtypes

category_main     object
category_sub      object
country           object
currency          object
goal_usd         float64
                  ...   
img_1275         float32
img_1276         float32
img_1277         float32
img_1278         float32
img_1279         float32
Length: 1295, dtype: object

In [52]:
obj_cols = train_df[final_features].select_dtypes(include="object").columns
print(obj_cols)

Index(['category_main', 'category_sub', 'country', 'currency', 'full_text'], dtype='object')


In [62]:
final_model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    loss_function="MultiClass",
    verbose=100
)

cat_features = train_df[features].select_dtypes(include="object").columns.tolist()

final_model.fit(
    train_df[final_features],
    train_df["target_band"],
    cat_features=cat_features
)

0:	learn: 1.5943344	total: 2.73s	remaining: 45m 30s
100:	learn: 1.1671056	total: 4m 29s	remaining: 39m 55s
200:	learn: 1.0581349	total: 8m 53s	remaining: 35m 21s
300:	learn: 0.9814668	total: 13m 18s	remaining: 30m 53s
400:	learn: 0.9189540	total: 17m 37s	remaining: 26m 19s
500:	learn: 0.8603911	total: 21m 54s	remaining: 21m 49s
600:	learn: 0.8097849	total: 26m 11s	remaining: 17m 23s
700:	learn: 0.7659818	total: 30m 28s	remaining: 12m 59s
800:	learn: 0.7267020	total: 34m 49s	remaining: 8m 39s
900:	learn: 0.6895495	total: 39m 9s	remaining: 4m 18s
999:	learn: 0.6571355	total: 43m 28s	remaining: 0us


CatBoostClassifier(depth=8, iterations=1000, learning_rate=0.03, loss_function='MultiClass', verbose=100)

In [63]:
preds = final_model.predict(
    test_df[final_features]
)

preds = preds.astype(int).flatten()

submission = pd.DataFrame({
    "project_id": test_df["project_id"],
    "target_band": preds
})

submission.to_csv(
    "/kaggle/working/submission_tfidf.csv",
    index=False
)

print("DONE!")

DONE!


In [24]:
print(train_df["image_path"].head())

0    images_224/P004133.jpg
1    images_224/P003029.jpg
2    images_224/P003262.jpg
3    images_224/P008745.jpg
4    images_224/P000423.jpg
Name: image_path, dtype: object


In [26]:
sample_path = train_df["image_path"].iloc[0]

print(sample_path)

full_path = os.path.join(
    CFG.image_dir,
    sample_path
)

print(full_path)

print(os.path.exists(full_path))

images_224/P004133.jpg
/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/images_224/images_224/P004133.jpg
False


In [34]:
CFG.image_dir = "/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/images"

In [35]:
sample_path = train_df["image_path"].iloc[0]

full_path = os.path.join(
    CFG.image_dir,
    sample_path
)

print(full_path)
print(os.path.exists(full_path))

/kaggle/input/competitions/ann-2026-multimodal-challenge-crowdfunding-outcome-prediction/multimodal_competition/processed/images/images_224/P004133.jpg
True
